# Week 04 ? Task 01: Rebuild nine charts, then take them apart

This notebook contains all nine Part A sub-tasks. Each case rebuilds the published claim, explains the misleading mechanism, and plots an honest redesign.

---

## Sub-task 1: Case 01_orders_stringdate

# Case 1 ? ?Orders peak in Q1 and collapse every year?

Rebuild the published recipe, identify the sorting mechanism, and redesign it with a chronological index.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path.cwd()
if not (DATA / "case01_orders_stringdate.csv").exists():
    DATA = Path(r"D:\University\ML Lab\Week04")
plt.style.use("seaborn-v0_8-whitegrid")

try:
    import tslib
    audit_index = tslib.audit_index
    banked_figsize = tslib.banked_figsize
except ModuleNotFoundError:
    # The course helper is not installed in this environment; keep the same API.
    def audit_index(index):
        idx = pd.DatetimeIndex(index)
        expected = pd.date_range(idx.min(), idx.max(), freq=pd.infer_freq(idx) or "D") if len(idx) else idx
        return {
            "sorted": idx.is_monotonic_increasing,
            "duplicates": int(idx.duplicated().sum()),
            "gaps": int(len(expected.difference(idx))) if len(idx) else 0,
            "n": len(idx), "start": idx.min(), "end": idx.max()
        }

    def banked_figsize(width=10):
        return (width, width / 1.618)

In [ ]:
df = pd.read_csv(DATA / "case01_orders_stringdate.csv")
published = df.sort_values("month")
assert len(published) == 24
changes = published["orders"].diff().dropna()
print("Self-check: points =", len(published), "drops =", changes[changes < 0].round(0).astype(int).tolist())
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(published["month"], published["orders"], marker="o")
ax.set(title="Published rebuild: orders sorted as strings", xlabel="Month (lexicographic order)", ylabel="Orders (count)")
ax.tick_params(axis="x", rotation=70)
plt.show()

**Conclusion and decoy.** The published line invites ?orders collapse every year after Q1.? A first objection might be that the declines are just unusually large random observations, but the exact drops (958 and 978) are produced by the plotted order, not by a data anomaly.

**Mechanism.** `month` is a string, so sorting puts `2024-10`, `2024-11`, and `2024-12` before `2024-2`; the apparent drops are transitions between non-adjacent calendar months. Parsing changes the affected row order while leaving all 24 values unchanged.

In [ ]:
df["month_date"] = pd.to_datetime(df["month"])
df = df.sort_values("month_date")
print("Index audit:", audit_index(pd.DatetimeIndex(df["month_date"])))
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(df["month_date"], df["orders"], marker="o", color="tab:blue")
ax.set(title="Orders in chronological order", xlabel="Month", ylabel="Orders (count)")
plt.tight_layout(); plt.show()

---

## Sub-task 2: Case 02_shop_closed

# Case 2 ? ?Sales grew steadily, 22%, across the quarter?

Rebuild the row-number chart, audit the calendar, and show the closure explicitly.

In [ ]:
df = pd.read_csv(DATA / "case02_shop_closed.csv", parse_dates=["date"])
y = df["sales_pkr_k"]
x = np.arange(len(df))
coef = np.polyfit(x, y, 1)
trend = np.polyval(coef, x)
print(f"Self-check: {len(df)} points; first = {y.iloc[0]}, last = {y.iloc[-1]}, rise = {(y.iloc[-1]/y.iloc[0]-1)*100:.1f}%")
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(x, y, label="Sales")
ax.plot(x, trend, "--", label="Least-squares trend")
ax.set(title="Published rebuild: row number hides calendar time", xlabel="Row number", ylabel="Sales (PKR k)")
ax.legend(); plt.show()

**Conclusion and decoy.** The chart invites ?sales rose steadily by 22%.? The first objection might be that the trend line is mathematically invalid; it is not?the regression is correctly fitted to the 69 observed rows.

**Mechanism.** The dates span 90 calendar days but only 69 rows: 21 consecutive dates (3?23 February) are absent because the shop was closed. Row number compresses that closure to a zero-width gap and makes the endpoints look like a continuous quarter. The missing period is data, not evidence of zero sales.

In [ ]:
df = df.set_index("date").sort_index()
full = df.reindex(pd.date_range(df.index.min(), df.index.max(), freq="D"))
print("Index audit:", audit_index(df.index))
print("Missing calendar days:", int(full["sales_pkr_k"].isna().sum()))
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(full.index, full["sales_pkr_k"], marker="o", color="tab:blue")
ax.set(title="Sales with the 21-day closure shown as a gap", xlabel="Date", ylabel="Daily sales (PKR k; observed rows)")
plt.tight_layout(); plt.show()

---

## Sub-task 3: Case 03_bandwidth_levels

# Case 3 ? ?Peak load jumped 43% overnight on 1 July?

Rebuild the daily sum, then use an aggregation appropriate for a concurrent level.

In [ ]:
df = pd.read_csv(DATA / "case03_bandwidth_levels.csv", parse_dates=["ts"]).set_index("ts").sort_index()
published = df["concurrent_users"].resample("D").sum()
print("Self-check: June mean =", round(published.loc["2025-06"].mean()), "July mean =", round(published.loc["2025-07"].mean()), "step =", round((published.loc["2025-07"].mean()/published.loc["2025-06"].mean()-1)*100), "%")
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(published.index, published, color="tab:orange")
ax.set(title="Published rebuild: daily sum of concurrent users", xlabel="Day", ylabel="Concurrent users per day (sum)")
plt.show()

**Conclusion and decoy.** The chart invites ?peak load jumped 43% overnight.? A first objection could be a missing-data problem, and there is one: the hourly index has irregular coverage. But the 43% step remains even when the observed counts are considered; it is mainly the aggregation choice.

**Mechanism.** `concurrent_users` is a level measured at an instant, not a flow to be accumulated. Daily `sum()` multiplies the level by the number of observations; June has a mean of about 8,782 summed units/day and July about 12,543, while the hourly level means are about 536 and 523. The apparent step is not a 43% increase in peak load.

In [ ]:
print("Index audit:", audit_index(df.index))
daily_mean = df["concurrent_users"].resample("D").mean()
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(daily_mean.index, daily_mean, marker=".", color="tab:blue")
ax.set(title="Typical concurrent load by day", xlabel="Day", ylabel="Concurrent users (daily mean; level)")
plt.show()

---

## Sub-task 4: Case 04_intervention_lag

# Case 4 ? ?The new checkout flow started working on 7 September?

Rebuild the trailing mean and show why its minimum is not an intervention timestamp.

In [ ]:
df = pd.read_csv(DATA / "case04_intervention_lag.csv", parse_dates=["date"]).set_index("date").sort_index()
rolling = df["conversion_pct"].rolling(31, min_periods=31).mean()
print("Self-check: trailing-mean minimum =", rolling.idxmin().date())
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(df.index, df["conversion_pct"], color="0.65", label="Raw daily conversion")
ax.plot(rolling.index, rolling, color="tab:red", linewidth=2, label="31-day trailing rolling mean")
ax.axvline(rolling.idxmin(), color="tab:red", linestyle=":", label="Smoothed minimum: 2025-09-07")
ax.set(title="Published rebuild: smoothed turning point", xlabel="Date", ylabel="Conversion (%)")
ax.legend(); plt.show()

**Conclusion and decoy.** The chart invites ?the new checkout flow started working on 7 September.? The obvious objection is that a rolling mean smooths noise; that is true but insufficient: the key issue is its directionality and lag.

**Mechanism.** A 31-day *trailing* mean contains the preceding 30 days. Its minimum is 2025-09-07, whereas the raw series minimum is 2025-09-03; a trailing window can only identify a delayed feature of the series, not an exact intervention date.

In [ ]:
print("Index audit:", audit_index(df.index))
raw_min = df["conversion_pct"].idxmin()
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(df.index, df["conversion_pct"], color="0.65", label="Raw daily conversion")
ax.plot(rolling.index, rolling, color="tab:blue", label="31-day trailing rolling mean")
ax.axvline(raw_min, color="black", linestyle="--", label=f"Raw minimum: {raw_min.date()}")
ax.axvline(rolling.idxmin(), color="tab:red", linestyle=":", label=f"Smoothed minimum: {rolling.idxmin().date()}")
ax.set(title="Conversion: raw observations and explicitly lagged smoothing", xlabel="Date", ylabel="Conversion (%)")
ax.legend(); plt.show()

---

## Sub-task 5: Case 05_revenue_window

# Case 5 ? ?Revenue has fallen for six straight months?

Rebuild the specified seven-month window, then show the complete history so the selection is visible.

In [ ]:
df = pd.read_csv(DATA / "case05_revenue_window.csv")
selected = df[df["month"].between("2025-05", "2025-11")]
assert len(selected) == 7
print(f"Self-check: {len(selected)} points; {selected.revenue_pkr_m.iloc[0]} down to {selected.revenue_pkr_m.iloc[-1]} = {(selected.revenue_pkr_m.iloc[-1]/selected.revenue_pkr_m.iloc[0]-1)*100:.1f}%")
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(selected["month"], selected["revenue_pkr_m"], marker="o")
ax.set(title="Published rebuild: selected May?November 2025 window", xlabel="Month", ylabel="Revenue (PKR m)")
ax.set_ylim(bottom=0); plt.show()

**Conclusion and decoy.** The chart invites ?revenue has fallen for six straight months.? The first objection might be that seven observations are too few; the recipe actually contains seven points, so that objection does not establish an error.

**Mechanism.** This is a window-selection problem: the seven-month slice is monotone, but the 36-month history is not. The selected window begins at the full-series maximum (158.8 in 2025-05), so the apparent sustained fall is incomplete context rather than a truncated axis.

In [ ]:
df["month_date"] = pd.to_datetime(df["month"])
print("Index audit:", audit_index(pd.DatetimeIndex(df["month_date"])))
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(df["month_date"], df["revenue_pkr_m"], marker="o", color="tab:blue")
ax.axvspan(pd.Timestamp("2025-05-01"), pd.Timestamp("2025-11-30"), color="tab:orange", alpha=.2, label="Published window")
ax.set(title="Complete revenue history with the published window marked", xlabel="Month", ylabel="Revenue (PKR m)")
ax.set_ylim(bottom=0); ax.legend(); plt.show()

---

## Sub-task 6: Case 06_monthly_partial

# Case 6 ? ?Orders collapsed in October?

Rebuild the monthly totals, audit coverage, and separate the incomplete October from the rate of orders per observed day.

In [ ]:
df = pd.read_csv(DATA / "case06_monthly_partial.csv", parse_dates=["date"]).set_index("date").sort_index()
monthly = df["orders"].resample("MS").sum()
print("Self-check: September =", monthly.loc["2025-09-01"], "October =", monthly.loc["2025-10-01"], "ratio =", round(monthly.loc["2025-10-01"]/monthly.loc["2025-09-01"]*100), "%")
fig, ax = plt.subplots(figsize=banked_figsize())
ax.bar(monthly.index, monthly.values, width=20, color="tab:orange")
ax.set(title="Published rebuild: monthly order totals", xlabel="Month", ylabel="Orders per month (sum)")
plt.show()

**Conclusion and decoy.** The chart invites ?orders collapsed in October.? A first objection might be that the bars have a bad baseline; they do not?the y-axis starts at zero.

**Mechanism.** October is partial: it contains 11 observed days versus 30 in September. The total falls to 4,041 simply because the observation window ends on 11 October; the chart is incomplete, not evidence of a monthly demand collapse.

In [ ]:
counts = df["orders"].resample("MS").count()
mean_daily = df["orders"].resample("MS").mean()
print("Index audit:", audit_index(df.index))
print("Observed days by month:", counts.to_dict())
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
colors = ["tab:blue" if n == counts.max() else "tab:orange" for n in counts]
ax1.bar(monthly.index, monthly.values, width=20, color=colors)
ax1.set(title="Monthly totals with incomplete October flagged", ylabel="Orders per month (sum)")
ax2.bar(mean_daily.index, mean_daily.values, width=20, color="tab:green")
ax2.set(title="Observed daily rate", xlabel="Month", ylabel="Orders per observed day (mean)")
plt.tight_layout(); plt.show()

---

## Sub-task 7: Case 07_stacked_products

# Case 7 ? ?Product D has been erratic since mid-2024?

Rebuild the stacked area chart, then isolate the product series and compare variation.

In [ ]:
df = pd.read_csv(DATA / "case07_stacked_products.csv")
products = ["product_a", "product_b", "product_c", "product_d"]
total = df[products].sum(axis=1)
print("Self-check: total envelope =", round(total.min(), 1), "to", round(total.max(), 1))
fig, ax = plt.subplots(figsize=banked_figsize())
ax.stackplot(df["month"], *(df[c] for c in products), labels=products)
ax.set(title="Published rebuild: stacked product areas", xlabel="Month", ylabel="Product value (stacked)")
ax.legend(loc="upper left"); plt.show()

**Conclusion and decoy.** The chart invites ?Product D became erratic after mid-2024.? The first objection might be that the total envelope is moving; that is not a defense of D, because a stack hides each component?s own movement.

**Mechanism.** Product D has a range of only 94.0?106.3 and standard deviation about 3.6. Product B has the large range (83.9?250.6) and standard deviation about 35.6. The top boundary and the changing thickness of lower bands make the stacked display visually attribute B?s variability to D.

In [ ]:
print("Index audit:", audit_index(pd.to_datetime(df["month"])))
print(df[products].agg(["min", "max", "std"]).round(1))
fig, ax = plt.subplots(figsize=banked_figsize())
for c in products:
    ax.plot(df["month"], df[c], marker="o", label=c)
ax.set(title="Product series shown independently", xlabel="Month", ylabel="Product value (not stacked)")
ax.legend(); ax.tick_params(axis="x", rotation=60); plt.tight_layout(); plt.show()

---

## Sub-task 8: Case 08_channel_attrib

# Case 8 ? ?Paid search drives 43% of our customers?

Rebuild the four-slice pie, test the part-to-whole conditions, and replace overlapping touch counts with exclusive combinations.

In [ ]:
df = pd.read_csv(DATA / "case08_channel_attrib.csv")
touch_cols = ["touch_search", "touch_social", "touch_email", "touch_direct"]
shares = df[touch_cols].sum() / len(df) * 100
print("Published shares (%):", shares.round(1).to_dict(), "sum =", round(shares.sum(), 3))
fig, ax = plt.subplots(figsize=banked_figsize(7))
ax.pie(shares.values, labels=[c.replace("touch_", "") for c in touch_cols], autopct="%.1f%%")
ax.set_title("Published rebuild: overlapping touch counts normalized as a pie")
plt.show()

**Conclusion and decoy.** The pie invites ?paid search drives 43% of customers.? A first objection might be rounding; rounding cannot explain the problem because the exact shares sum to 130.975%, not 100%.

**Mechanism.** The parts are not exclusive: one customer can have multiple `True` touches. The four counts describe channel reach, not mutually exclusive customer shares. Therefore the whole is not partitioned and a pie silently renormalizes the data.

In [ ]:
# Exclusive combinations are valid parts of the customer whole.
combo = df[touch_cols].astype(int).astype(str).agg("".join, axis=1)
combo_counts = combo.value_counts().sort_values(ascending=False)
combo_share = combo_counts / len(df) * 100
assert abs(combo_share.sum() - 100) < 1e-6
labels = combo_counts.index.map(lambda s: " + ".join(c.replace("touch_", "") for c, bit in zip(touch_cols, s) if bit == "1") or "No touch")
print("Exclusive combinations:", len(combo_counts), "categories; shares sum =", combo_share.sum())
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(labels, combo_share, color="tab:blue")
ax.set(title="Customers by exclusive touch combination", xlabel="Exclusive combination", ylabel="Customers (%)")
ax.tick_params(axis="x", rotation=70); plt.tight_layout(); plt.show()

---

## Sub-task 9: Case 09_premium_share

# Case 9 ? ?Premium is now 38% of revenue ? a record?

Rebuild the share line, then plot the numerator and denominator on a common scale so the denominator effect is visible.

In [ ]:
df = pd.read_csv(DATA / "case09_premium_share.csv")
df["share_pct"] = df["premium_revenue_pkr_m"] / df["total_revenue_pkr_m"] * 100
print(f"Self-check: share {df.share_pct.iloc[0]:.1f}% to {df.share_pct.iloc[-1]:.1f}%")
fig, ax = plt.subplots(figsize=banked_figsize())
ax.plot(df["quarter"], df["share_pct"], marker="o")
ax.annotate(f"{df.share_pct.iloc[-1]:.1f}%", (df.quarter.iloc[-1], df.share_pct.iloc[-1]), xytext=(-35, 10), textcoords="offset points")
ax.set(title="Published rebuild: premium share", xlabel="Quarter", ylabel="Premium revenue share (%)")
ax.set_ylim(bottom=0); plt.show()

**Conclusion and decoy.** The chart invites ?premium is now 38% of revenue?a record.? A first objection might be that the percentage calculation is wrong; it is arithmetically correct.

**Mechanism.** The denominator shrank from PKR 1,000m to PKR 600m while premium revenue fell from PKR 240.0m to PKR 228.0m. The rising share is therefore a denominator effect; 228.0m is not a record premium revenue value.

In [ ]:
print("Index audit:", audit_index(pd.period_range("2024Q1", periods=len(df), freq="Q").to_timestamp()))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
ax1.plot(df["quarter"], df["total_revenue_pkr_m"], marker="o", label="Total revenue")
ax1.plot(df["quarter"], df["premium_revenue_pkr_m"], marker="o", label="Premium revenue")
ax1.set(title="Absolute revenue values", ylabel="Revenue (PKR m)"); ax1.legend()
ax2.plot(df["quarter"], df["share_pct"], marker="o", color="tab:purple")
ax2.set(title="Share (denominator-dependent)", xlabel="Quarter", ylabel="Premium revenue share (%)")
ax2.set_ylim(bottom=0); plt.tight_layout(); plt.show()